<h3><b>Introduction</b></h3>
<p> The goal of this project is to take the publically available AQMRG weather data from 2013 to 2017 and apply machine learning techniques to see if we can predict the amount of PM2.5 concentration in the air given other environmental features. This is a project I am working on during my free time applying some of the machine learning algorithms I have learned. I hope to come up with a predictive model with a high accuracy and a very low Root Mean Square Error (RMSE).</p>
<br>
<h5><b>Dataset Information</b></h5>
<p>This data set includes hourly air pollutants data from 12 AQMRG sensor network. The air-quality data are from the AQMRG Municipal Environmental Monitoring Center. The meteorological data in each air-quality site are matched with the nearest weather station from the AQMRG Local Weather Station. The time period is from March 1st, 2013 to February 28th, 2017. Missing data are denoted as NA.</p>
<br>
<h5><b>Project workflow</b></h5>
<p>This project utilizes the Supervised Machine Learning algorithms from python’s Scikit-learn library. The model we hope to succeed in training is a regression model and below are the steps we will go through in this jupyter notebook for this project:</p>
<ul>
    <li>Import the neccessary libraries and loading the data</li>
    <li>Data preprocessing</li>
    <li>Exploratory Data Ananlysis</li>
    <li>Model training and Evaluation</li>
    <li>Saving the model</li>
</ul>
<br>
<h5><b>Import libraries and loading data</b></h5>

In [1]:
import joblib
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

In [4]:
# The original model was incompatible with this scikit-learn version.
# We are retraining it using the data processed in previous cells.
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
import joblib
import numpy as np

# Retrain the model (using the parameters found in the notebook)
# Note: xtrain, ytrain must be defined in previous cells.
# This assumes the notebook has been run up to the Ensemble methods section.

params = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'max_features': ['sqrt', 'log2', None], # 'auto' is deprecated in 1.1+
    'min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'min_samples_leaf': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'learning_rate': np.linspace(0.1, 1, 10)
}

gradient_boosting = GradientBoostingRegressor()
gboost_search = RandomizedSearchCV(gradient_boosting, params, n_jobs=-1, cv=5, verbose=1)
gboost_search.fit(xtrain, ytrain)

# Save the new model to the expected location
joblib.dump(gboost_search.best_estimator_, '_Desktop_pm25_model.pkl')
regression_model = gboost_search.best_estimator_
print('New model trained and saved as gboost_model_PM2.5.pkl')

NameError: name 'xtrain' is not defined

<h3><b>Data Preprocessing</b></h3>

In [ ]:
# Preprocessing: Convert timestamp and extract time features
data['Timestamp (EAT)'] = pd.to_datetime(data['Timestamp (EAT)'])
data['year'] = data['Timestamp (EAT)'].dt.year
data['month'] = data['Timestamp (EAT)'].dt.month
data['day'] = data['Timestamp (EAT)'].dt.day
data['hour'] = data['Timestamp (EAT)'].dt.hour

# Drop relay-specific columns but keep others for mapping
cols_to_drop = ['ID', 'Device ID', 'Latitude', 'Longitude']
data = data.drop([c for c in cols_to_drop if c in data.columns], axis=1)

# Handle CO2 imputation before setting index
import numpy as np
if 'CO2' in data.columns:
    data['CO2'] = data['CO2'].replace(-1.0, np.nan)
    data['CO2'] = data['CO2'].fillna(data['CO2'].mean())

data.set_index('Timestamp (EAT)', inplace=True)
data.head()


In [ ]:
# check for duplicated values and null values
print('Are there any duplicated values in our data ? : {}\n'.format(data.duplicated().any()))
print('The total number of null values in each colum:')
display(data.isnull().sum())

In [ ]:
# Handling missing values in CO2 (-1.0 values)
import numpy as np
if 'CO2' in data.columns:
    data['CO2'] = data['CO2'].replace(-1.0, np.nan)
    data['CO2'] = data['CO2'].fillna(data['CO2'].mean())

print("Missing values after handling CO2:")
print(data.isnull().sum())

In [ ]:
if 'wd' in data.columns:
    plt.figure(figsize=(12,5))
    sns.countplot(data.wd)
    plt.title('Wind direction count')
    plt.show()
else:
    print('Wind direction (wd) not available in this dataset.')


In [ ]:
data = data.dropna(subset=['PM2.5'])
data.columns = [c.replace('.', '_') for c in data.columns]
data.head()


<h3><b>Exploratory Data Analysis</b></h3>
<p>Before we start fitting a machine learning model on the data, we need to know much about the data by performing an Exploratory Data Analysis to gain insight from it. <abbr title="Exploratory Data Analysis">EDA</abbr> is simply describing the data by means of visualization. It involves asking questions about the data and answering them with the help of charts/graphs (graphical representation of the data). In this process, we will try to study the behavoir of the amount of pollutant (PM2.5 concentration) in the air and the relationship between other features. Below are some of the questions we will try to answer by analyzing the data, to know more about our dependent and independent variables:</p>
<ul>
    <li>what pattern does the amount of PM2.5 concentration in the air recorded in an hour follow
        for a daily time period ?</li>
    <li>In which month does the amount of PM2.5 contained in the air rises ?</li>
    <li>At what time of the day do we expect the amount of PM2.5 concentration in the
        air to be high ?</li>
    <li>In which direction does polluted air/wind mostly move ?</li>
    <li>How do the other environmental factors affect the amount of PM2.5 concentration
        in the air ?</li>
</ul>
<p>We now have our questions so let's just dive into our data and start finding and interpreting some results. But since we are going to take averages of the dependent variable, we shoul know the distribtution of the data before we do take averages.</p>

In [ ]:
plt.figure(figsize=(12,5))
sns.distplot(data['PM2.5'],bins=50)
plt.title('Distribution of the hourly recorded PM2.5 concetration in the air\nin Relay Data-AQMRG',
          fontsize=16)
plt.show()

<h5><b>what pattern does the amount of PM2.5 concentration in the air recorded in an hour follow
        for a daily time period ?</b></h5>

In [ ]:
# Daily average of PM2.5
target_col = 'PM2_5' if 'PM2_5' in data.columns else 'PM2.5'
daily_data = data[[target_col]].resample('D').median()
decomposition = seasonal_decompose(daily_data.dropna(), model='additive')

with plt.style.context('fivethirtyeight'):
    decomposition.trend.plot(figsize=(12,5), style='k-', linewidth=.9, legend=False)
    plt.xlabel('Date', fontsize=14)
    plt.ylabel(f'{target_col} concentration (ug/m^3)', fontsize=14)
    plt.title('Daily trend in PM2.5 concentration', fontsize=16)
    plt.show()


<h5><b>In which month does the amount of PM2.5 contained in the air rises ?</b></h5>

In [ ]:
monthly_data = data[['month','PM2.5']]
months = ['January','February','March','April','May','June','July',
         'August','September','October','November','December']
ordered_monthdf = pd.DataFrame(months,columns=['month'])
map_dict = {}
for i,j in enumerate(months):
    map_dict.setdefault(i+1,j)

monthly_data.month = monthly_data.month.map(map_dict)
monthly_average = monthly_data.groupby('month').median()
monthly_average = pd.merge(ordered_monthdf,monthly_average,left_on='month',right_index=True)
monthly_average = np.round(monthly_average,1)
monthly_average = monthly_average.set_index('month')

# plot the data
with plt.style.context('ggplot'):
    monthly_average.plot(figsize=(12,5),legend=False,kind='bar',linewidth=.9)
    plt.xlabel('Month',fontsize=14)
    plt.ylabel('PM2.5 concentration (ug/m^3)',fontsize=14)
    plt.title('Monthly average of the hourly recorded PM2.5 concentration in\nthe air in Relay Data-AQMRG',fontsize=16)
    plt.grid(axis='x')
    plt.tight_layout()
    plt.show()

<h5><b>At what time of the day do we expect the amount of PM2.5 concentration in the air to be high ?</b></h5>

In [ ]:
hourly_data = data[['hour','PM2.5']]
hrs = ['12 AM','1 AM','2 AM','3 AM','4 AM','5 AM','6 AM','7 AM','8 AM','9 AM','10 AM',
      '11 AM','12 PM','1 PM','2 PM','3 PM','4 PM','5 PM','6 PM','7 PM',
      '8 PM','9 PM','10 PM','11 PM']
hour_dict = {}
for i,j in enumerate(hrs):
    hour_dict.setdefault(i,j)
    
hourly_data = hourly_data.groupby('hour').median().reset_index()
hourly_data.hour = hourly_data.hour.map(hour_dict)
hourly_data = hourly_data.set_index('hour')

# plot the data
with plt.style.context('ggplot'):
    hourly_data.plot(figsize=(12,8),legend=False,kind='barh',linewidth=.9)
    plt.ylabel('Hours',fontsize=14)
    plt.xlabel('PM2.5 concentration (ug/m^3)',fontsize=14)
    plt.title('Average recorded PM2.5 concentration in the air in Relay Data-AQMRG\nby the hour of the day',fontsize=16)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()

<h5><b>In which direction does polluted air/wind mostly move ?</b></h5>

In [ ]:
wind_dir = data[['wd','PM2.5']]
wind_dir = wind_dir.groupby('wd').median()

# plot the data
with plt.style.context('ggplot'):
    wind_dir.plot(figsize=(12,5),legend=False,kind='bar',linewidth=.9)
    plt.xlabel('Wind direction',fontsize=14)
    plt.ylabel('PM2.5 concentration (ug/m^3)',fontsize=14)
    plt.title('Average hourly recorded PM2.5 concentration in the air in Relay Data-AQMRG\ngrouped by wind direction',fontsize=16)
    plt.grid(axis='x')
    plt.tight_layout()
    plt.show()

<h5><b>How do the other environmental factors affect the amount of PM2.5 concentration in the air ?</b></h5>

In [ ]:
# let's try and visualize the relationships between the features of the data
plt.figure(figsize=(13,9))
correlation_data = data.select_dtypes(include=[np.number])
sns.heatmap(correlation_data.corr(),cmap=plt.cm.Reds,annot=True)
plt.title('Heatmap displaying the correlation matrix of the variables',fontsize=16)
plt.show()


<h3><b>Model Training and Evaluation</b></h3>
<br>
<h5><b>check for multicollinearity among variables and fit a regression model using statsmodels</b></h5>

In [ ]:
# no extra columns to drop since we preprocessed earlier,
# but to maintain notebook logic, we ensures we have the right data
newdata = data.copy()
if 'Timestamp (EAT)' in newdata.columns:
    newdata = newdata.drop(['Timestamp (EAT)'], axis=1)

newdata = data.drop(cols_to_drop,axis=1)

# calculate the variance inflation factor of each feature and detect multicollinearity
cons_data = sm.tools.add_constant(newdata)
series_before = pd.Series([variance_inflation_factor(cons_data.values,i) for i in range(cons_data.shape[1])],
                         index=cons_data.columns)
series_before

In [ ]:
if 'DEWP' in newdata.columns:
    newdata = newdata.drop('DEWP',axis=1)
cons_data2 = sm.tools.add_constant(newdata)
series_after = pd.Series([variance_inflation_factor(cons_data2.values,i) for i in range(cons_data2.shape[1])],
                         index=cons_data2.columns)
series_after


In [ ]:
newdata.columns = [c.replace('.', '_') for c in newdata.columns]

# PM2.5 is skewed to the right so we log transform the values to normalize the distribution
target_col = 'PM2_5'
newdata[target_col] = np.log(newdata[target_col])


In [ ]:
# fit the regression model dynamically
features = [c for c in newdata.columns if c != 'PM2_5']
formula_str = 'PM2_5 ~ ' + ' + '.join(features)
mul_reg = formula.ols(formula=formula_str, data=newdata).fit()
mul_reg.summary()


<p>The OLS model from statsmodels gives us an accuracy of 71% (0.712) which is not satisfactory for prediction. So we move on to fit a linear regression model from the scikit-learn library.</p>
<br>
<h5><b>fitting a linear regression model with sklearn.linear_model.LinearRegression()</b></h5>

In [ ]:
# we split the data into predictor variables and Outcome variable
X = newdata.drop('PM2_5',axis=1)
y = newdata['PM2_5']

# we need to scale or normalize the predictor variables since they are not on the same
# scale and some of their distributions are skewed.
X_scaled =  preprocessing.scale(X)
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)
X_scaled.dropna(inplace=True)
# print the scaled predictor variables.
X_scaled.head()

In [ ]:
# we now split out data into train and test data
X_train,X_test,y_train,y_test = train_test_split(X_scaled,y,test_size=.2,random_state=0)

# instantiate the linear regression model
lin_model = LinearRegression()
lin_model.fit(X_train,y_train)   # fit the model

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(lin_model.score(X_train,y_train)))
print('Score on test data: {}'.format(lin_model.score(X_test,y_test)))

In [ ]:
prediction = lin_model.predict(X_test)
mse = mean_squared_error(y_test,prediction)
accuracy = r2_score(y_test,prediction)

print('Mean Squared Error: {}\n'.format(mse))
print('Overall model accuracy: {}'.format(accuracy))

<p>The model accuracy for the LinearRegression() is no better than that of the statsmodels. They all give the same accuracy is not better for making predictions. We now move on to fit other models by using the ensemble methods</p>
<br>
<br>
<h3><b>Ensemble methods</b></h3>
<p>For ensemble methods (DecisionTreeRegressor,RandomForestRegressor,and GradientBoostingRegressor),we include the pressure and rain features and we won't scale the predictor variables,neither would we log transform the outcome variable. At this part, model fitting and hyper-parameter tunning will be done at the same time. Instead of fitting the model with single parameters, we will straight away perform the grid search with multiple values for a parameter and find the best parameters for fitting the model on our data to get a satisfactory accuracy.</p>

In [ ]:
ensemble_data = data.copy()

# we split the data into predictor variables and Outcome variable
X = ensemble_data.drop('PM2.5',axis=1)
y = ensemble_data['PM2.5']


In [ ]:
xtrain,xtest,ytrain,ytest = train_test_split(X,y,test_size=.2)

### DecisionTreeRegressor
we will now fit a decision tree regression model on the data and tune some of its parameters to increase the accuracy.

In [ ]:
# we go ahead to use the ensemble methods as the LinearRegression model has a low accuracy
# on both the test and train data.
decision_tree = DecisionTreeRegressor(max_depth=5,
                                     max_features='auto',
                                     min_samples_split=3,
                                     min_samples_leaf=2)
decision_tree.fit(xtrain,ytrain)

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(decision_tree.score(xtrain,ytrain)))
print('Score on test data: {}\n'.format(decision_tree.score(xtest,ytest)))

tree_pred = decision_tree.predict(xtest)
tree_mse = mean_squared_error(ytest,tree_pred)
tree_accuracy = r2_score(ytest,tree_pred)

print('Root Mean Squared Error: {}\n'.format(np.sqrt(tree_mse)))
print('Overall model accuracy: {}'.format(tree_accuracy))

In [ ]:
# We now tune the parameters of the model to see if we can increase the accuracy
params = {'max_depth':[3,4,5,6,7],
         'max_features':['auto','sqrt','log2'],
         'min_samples_split':[2,3,4,5,6,7,8,9,10],
         'min_samples_leaf':[2,3,4,5,6,7,8,9,10]}

tree = DecisionTreeRegressor()

# initialize the grid search for the best parameters
tree_search = GridSearchCV(tree,param_grid=params,
                          n_jobs=-1,cv=5)

tree_search.fit(xtrain,ytrain)   # fit the model

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(tree_search.score(xtrain,ytrain)))
print('Score on test data: {}\n'.format(tree_search.score(xtest,ytest)))
print('Best parameters found:')
display(tree_search.best_params_)

tree_search_pred = tree_search.predict(xtest)
tree_search_mse = mean_squared_error(ytest,tree_search_pred)
tree_search_accuracy = r2_score(ytest,tree_search_pred)

print('Root Mean Squared Error: {}\n'.format(np.sqrt(tree_search_mse)))
print('Overall model accuracy: {}'.format(tree_search_accuracy))

### RandomForestRegressor

we now fit a random forest regression model on the data to see if we would get a better accuracy results than that of the decision tree regression model.

In [ ]:
# instantiate the RandomForestRegressor model and fit the model on the training data
forest = RandomForestRegressor(n_estimators=100,
                              max_depth=7,
                              max_features='auto',
                              min_samples_split=7,
                              min_samples_leaf=3)

forest.fit(xtrain,ytrain)

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(forest.score(xtrain,ytrain)))
print('Score on test data: {}\n'.format(forest.score(xtest,ytest)))

forest_pred = forest.predict(xtest)
forest_mse = mean_squared_error(ytest,forest_pred)
forest_accuracy = r2_score(ytest,forest_pred)

print('Root Mean Squared Error: {}\n'.format(np.sqrt(forest_mse)))
print('Overall model accuracy: {}'.format(forest_accuracy))

In [ ]:
# we now tune the parameters of the RandomForestRegressor model using RandomizedSearchCV to 
# find the best parameters and increase the accuracy of the model

params['n_estimators'] = [100,200,300,400,500]

# instantiate the model
random_forest = RandomForestRegressor()

# perform the grid search for the best parameters
forest_search = RandomizedSearchCV(random_forest,params,n_jobs=-1,
                                   cv=5,verbose=2)
forest_search.fit(xtrain,ytrain)

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(forest_search.score(xtrain,ytrain)))
print('Score on test data: {}\n'.format(forest_search.score(xtest,ytest)))
print('Best parameters found:')
display(forest_search.best_params_)

forest_search_pred = forest_search.predict(xtest)
forest_search_mse = mean_squared_error(ytest,forest_search_pred)
forest_search_accuracy = r2_score(ytest,forest_search_pred)

print('Root Mean Squared Error: {}\n'.format(np.sqrt(forest_search_mse)))
print('Overall model accuracy: {}'.format(forest_search_accuracy))

### GradientBoostingRegressor

we now fit a gradient boosting regression model on the data to see if we would get a better accuracy results than that of the decision tree and random forest regression model and also minimize the error.

In [ ]:
# instantiate the GradientBoostingRegressor model and fit the model on the training data
grad_boost = GradientBoostingRegressor(n_estimators=100,
                                      max_depth=7,
                                      max_features='auto',
                                      min_samples_split=7,
                                      min_samples_leaf=3,
                                      learning_rate=0.1)

grad_boost.fit(xtrain,ytrain)

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(grad_boost.score(xtrain,ytrain)))
print('Score on test data: {}\n'.format(grad_boost.score(xtest,ytest)))

gboost_pred = grad_boost.predict(xtest)
gboost_mse = mean_squared_error(ytest,gboost_pred)
gboost_accuracy = r2_score(ytest,gboost_pred)

print('Root Mean Squared Error: {}\n'.format(np.sqrt(gboost_mse)))
print('Overall model accuracy: {}'.format(gboost_accuracy))

In [ ]:
# we now tune the parameters of the GradientBoostingRegressor model using RandomizedSearchCV to 
# find the best parameters and increase the accuracy of the model

params['learning_rate'] = np.linspace(0.1,1,10)

# instantiate the model
gradient_boosting = GradientBoostingRegressor()

# perform the grid search for the best parameters
gboost_search = RandomizedSearchCV(gradient_boosting,params,n_jobs=-1,
                                   cv=5,verbose=2)
gboost_search.fit(xtrain,ytrain)

In [ ]:
# we now score the model
print('Score on train data: {}\n'.format(gboost_search.score(xtrain,ytrain)))
print('Score on test data: {}\n'.format(gboost_search.score(xtest,ytest)))
print('Best parameters found:')
display(gboost_search.best_params_)

gboost_search_pred = gboost_search.predict(xtest)
gboost_search_mse = mean_squared_error(ytest,gboost_search_pred)
gboost_search_accuracy = r2_score(ytest,gboost_search_pred)

print('Root Mean Squared Error: {}\n'.format(np.sqrt(gboost_search_mse)))
print('Overall model accuracy: {}'.format(gboost_search_accuracy))

In [ ]:
# we now use the best model (GradientBoostingRegressor model) to predict the PM2.5 
# concetration and compare it to the actual PM2.5 recorded in the data by means of
# visualization

compare_data = pd.DataFrame({'dates':data['date'],
                            'Actual PM2.5':y,
                            'Predicted PM2.5':gboost_search.predict(X.values)})

compare_data.set_index('dates',inplace=True)
compare_data['Predicted PM2.5'] = np.round(compare_data['Predicted PM2.5'],1)

# let's plot the daily averages of the Actual PM10 and the predicted PM2.5 concentration.
compare_data = compare_data.resample('D').mean()

with plt.style.context('fivethirtyeight'):
    plt.figure(figsize=(12,5))
    plt.scatter(compare_data.index,compare_data['Actual PM2.5'],s=15,label='Actual PM2.5',
               alpha=.6)
    plt.scatter(compare_data.index,compare_data['Predicted PM2.5'],s=15,label='Predicted PM2.5',
               alpha=.6)
    plt.legend()
    plt.title('Evaluating the GradientBoostingRegressor model\n(model accuracy = 93%)',
             fontsize=18)
    plt.xlabel('period',fontsize=15)
    plt.ylabel('PM2.5 concentration',fontsize=15)
    plt.show()

<h5><b>Residuals analysis</b></h5>
<p>Now that we have successfully trained a regression model that predicts the amount of PM2.5 concetration in the air with a 93% accuracy given other environmental features, we have to analyze the errors of prediction to see if the model satisfies the regression errors asumption. That is,the errors must be normally distributed and do not follow any pattern.</p>

In [ ]:
# calculate the errors
compare_data['Residuals'] = compare_data['Actual PM2.5'] - compare_data['Predicted PM2.5']

# make a scatter plot of the errors to see if they follow any pattern
with plt.style.context('ggplot'):
    plt.figure(figsize=(12,5))
    plt.scatter(compare_data.index,compare_data.Residuals,alpha=.7)
    plt.title('Residual scatter plot',fontsize=16)
    plt.ylabel('Errors',fontsize=15)
    plt.grid(axis='x')
    plt.show()

In [ ]:
# plot the histogram to see check the normality of the errors
plt.figure(figsize=(12,5))
sns.distplot(compare_data.Residuals,bins=50)
plt.title('Distribution of the residuals from the gradient boosting model',
         fontsize=16)
plt.show()

<p>As seen above, the residuals of the predictions follow no pattern and also have a normal distribution which satisfies the regression error assumptions. This proves that our model is accurate and good for further predictions.</p>
<br>
<h3><b>Saving and loading the model</b></h3>
<p>Now that the model has been successfully trained, the next thing is to save and be able to load it anytime we want to use it. To do this, we have to save it as a pickle file using the joblib module from sklearn.externals. The cells below shows you how to save your trained machine learning model and also load it anytime you want.</p>

In [ ]:
from sklearn.externals import joblib

In [ ]:
# save the model
import os
home = os.path.expanduser('~')
save_path = os.path.join(home, 'Desktop', 'pm25_model.pkl')
joblib.dump(gboost_search, save_path)
print(f'Model saved to {save_path}')


In [ ]:
# loading the model
home = os.path.expanduser('~')
load_path = os.path.join(home, 'Desktop', 'pm25_model.pkl')
if os.path.exists(load_path):
    regression_model = joblib.load(load_path)
    print(f'Model loaded from {load_path}')
else:
    print(f'Model file not found at {load_path}')


<br>
<br>
<br>
<h4><b>Project completed by: Prince Owusu</b></h4>
<p><a href="mailto:powusu381@gmail.com" target="_blank">Email</a> || <a href="https://www.linkedin.com/in/prince-owusu-356914198?lipi=urn%3Ali%3Apage%3Ad_flagship3_profile_view_base_contact_details%3B2NYoXqMHQKOMp0yWSME5mQ%3D%3D" target="_blank">LinkedIn</a> || <a href="https://twitter.com/iam_kwekhu" target="_blank">Twitter</a></p>